# 01 — Exploratory Data Analysis
Khám phá và trực quan hoá dữ liệu ban đầu.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Load dữ liệu
DATA_PATH = Path('../data/processed/youtube_comments_sentiment.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Đã load {len(df)} dòng từ {DATA_PATH}")
    print(f"\nColumns: {list(df.columns)}")
    df.head()
else:
    print(f"❌ Không tìm thấy file: {DATA_PATH}")
    print("Hãy chạy crawler và sentiment analyzer trước!")


## 1. Tổng quan dữ liệu

In [ ]:
# Thông tin cơ bản
print("=== SHAPE ===")
print(df.shape)
print("\n=== INFO ===")
print(df.info())
print("\n=== DESCRIBE ===")
df.describe()

## 2. Phân bố cảm xúc

In [ ]:
if 'sentiment' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Count plot
    sentiment_counts = df['sentiment'].value_counts()
    axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=['#22c55e', '#94a3b8', '#ef4444'])
    axes[0].set_title('Số lượng bình luận theo cảm xúc', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Cảm xúc')
    axes[0].set_ylabel('Số lượng')
    for i, v in enumerate(sentiment_counts.values):
        axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')
    
    # Pie chart
    axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%',
                colors=['#22c55e', '#94a3b8', '#ef4444'], startangle=90)
    axes[1].set_title('Tỷ lệ cảm xúc', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Thống kê
    print("\n=== PHÂN BỐ CẢM XÚC ===")
    print(sentiment_counts)
    print(f"\nTỷ lệ tiêu cực: {(sentiment_counts.get('negative', 0) / len(df) * 100):.1f}%")
else:
    print("❌ Không có cột 'sentiment' trong dữ liệu")

## 3. Phân tích độ tin cậy (Confidence)

In [ ]:
if 'confidence' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(df['confidence'], bins=30, color='#6366f1', edgecolor='black', alpha=0.7)
    axes[0].set_title('Phân bố độ tin cậy', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Confidence')
    axes[0].set_ylabel('Số lượng')
    axes[0].axvline(df['confidence'].mean(), color='red', linestyle='--', label=f'Mean: {df["confidence"].mean():.2f}')
    axes[0].legend()
    
    # Box plot by sentiment
    if 'sentiment' in df.columns:
        df.boxplot(column='confidence', by='sentiment', ax=axes[1])
        axes[1].set_title('Confidence theo cảm xúc', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Cảm xúc')
        axes[1].set_ylabel('Confidence')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== THỐNG KÊ CONFIDENCE ===")
    print(df['confidence'].describe())
else:
    print("❌ Không có cột 'confidence' trong dữ liệu")

## 4. Phân tích thời gian

In [ ]:
# Tìm cột thời gian
time_col = None
for col in ['published_at', 'published_time', 'crawled_at']:
    if col in df.columns:
        time_col = col
        break

if time_col:
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    df_valid = df.dropna(subset=[time_col])
    
    if not df_valid.empty:
        df_valid['date'] = df_valid[time_col].dt.date
        daily_counts = df_valid.groupby('date').size()
        
        plt.figure(figsize=(14, 5))
        plt.plot(daily_counts.index, daily_counts.values, marker='o', linewidth=2, markersize=4, color='#6366f1')
        plt.title('Số lượng bình luận theo thời gian', fontsize=14, fontweight='bold')
        plt.xlabel('Ngày')
        plt.ylabel('Số bình luận')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print(f"\n=== THỐNG KÊ THEO THỜI GIAN ===")
        print(f"Từ: {df_valid[time_col].min()}")
        print(f"Đến: {df_valid[time_col].max()}")
        print(f"Tổng số ngày: {len(daily_counts)}")
        print(f"Trung bình {daily_counts.mean():.1f} bình luận/ngày")
    else:
        print("❌ Không có dữ liệu thời gian hợp lệ")
else:
    print("❌ Không tìm thấy cột thời gian")

## 5. Top từ khóa nổi bật

In [ ]:
from collections import Counter
import re

# Tìm cột text
text_col = None
for col in ['text', 'description', 'title']:
    if col in df.columns:
        text_col = col
        break

if text_col:
    # Stopwords đơn giản
    stopwords = {'là', 'và', 'có', 'cho', 'của', 'một', 'những', 'được', 'trong', 'với', 
                 'này', 'đó', 'thì', 'khi', 'rồi', 'đang', 'như', 'quá', 'video', 
                 'comment', 'youtube', 'ai', 'vì', 'các', 'cái', 'để', 'mình', 
                 'bạn', 'ko', 'không', 'rất', 'chủ', 'đề'}
    
    # Extract words
    all_words = []
    for text in df[text_col].dropna().astype(str):
        words = re.findall(r'\b\w{3,}\b', text.lower())
        all_words.extend([w for w in words if w not in stopwords])
    
    # Top keywords
    word_freq = Counter(all_words).most_common(15)
    
    # Plot
    plt.figure(figsize=(12, 6))
    words, counts = zip(*word_freq)
    plt.barh(words[::-1], counts[::-1], color='#22d3ee')
    plt.title('Top 15 từ khóa nổi bật', fontsize=14, fontweight='bold')
    plt.xlabel('Tần suất')
    plt.tight_layout()
    plt.show()
    
    print("\n=== TOP TỪ KHÓA ===")
    for word, count in word_freq:
        print(f"{word}: {count}")
else:
    print("❌ Không tìm thấy cột text")

## 6. Tóm tắt

In [ ]:
print("=== TÓM TẮT DỮ LIỆU ===\n")
print(f"Tổng số bình luận: {len(df):,}")

if 'sentiment' in df.columns:
    print(f"Cảm xúc tích cực: {(df['sentiment'] == 'positive').sum():,} ({(df['sentiment'] == 'positive').mean() * 100:.1f}%)")
    print(f"Cảm xúc tiêu cực: {(df['sentiment'] == 'negative').sum():,} ({(df['sentiment'] == 'negative').mean() * 100:.1f}%)")
    print(f"Cảm xúc trung tính: {(df['sentiment'] == 'neutral').sum():,} ({(df['sentiment'] == 'neutral').mean() * 100:.1f}%)")

if 'confidence' in df.columns:
    print(f"\nĐộ tin cậy TB: {df['confidence'].mean() * 100:.1f}%")
    print(f"Độ tin cậy min: {df['confidence'].min() * 100:.1f}%")
    print(f"Độ tin cậy max: {df['confidence'].max() * 100:.1f}%")

print("\n✅ EDA hoàn thành!")